In [23]:
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns

from sklearn.compose import ColumnTransformer
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import PolynomialFeatures


In [24]:
df = pd.read_csv("data/cars.csv")

# using the 99th percentile removes most outliers
price_upper = df["price"].quantile(0.99)
df_clean = df[df["price"] <= price_upper].copy()

In [25]:
X_train, X_test = train_test_split(
    df_clean,
    test_size=0.30,
    random_state=42,
)

### Simple linear regression using age

The code below trains a simple linear regression model using `age` as the only predictor of `price`.

The model is evaluated on the test set using three regression performance metrics:

- `mae`: mean absolute error, or the average absolute difference between actual and predicted prices.
- `rmse`: root mean squared error, which penalizes larger prediction errors more heavily.
- `r2`: R-squared, which measures how much variation in price is explained by the model.

In [26]:
features = ["age"]

y_train = X_train["price"]
y_test = X_test["price"]

select_features = ColumnTransformer(
    transformers=[
        ("selected", "passthrough", features)
    ],
    remainder="drop"
)

pipe = Pipeline([
    ("select_features", select_features),
    ("model", LinearRegression())
])

pipe.fit(X_train, y_train)
preds = pipe.predict(X_test)

performance = pd.Series({
    "mae": mean_absolute_error(y_test, preds),
    "rmse": mean_squared_error(y_test, preds) ** 0.5,
    "r2": r2_score(y_test, preds),
})

prediction_summary = pd.DataFrame({
    "y_test": y_test,
    "preds": preds,
}).agg(["mean", "std", "median", "min", "max"])

print(performance)
print(prediction_summary)

mae      7914.964240
rmse    10083.714515
r2          0.401999
dtype: float64
              y_test         preds
mean    19318.045232  19339.033348
std     13039.824946   8243.202183
median  16591.000000  21465.769840
min      1000.000000  -7673.312303
max     65500.000000  33734.857058


### Interpreting the model performance

These numbers suggest that the `age`-only linear regression model has some predictive signal, but it is not a very strong pricing model.

The MAE is about `$7,915`, which means the model is off by about `$7,915` on average. The RMSE is higher, about `$10,084`, which means some predictions have larger errors. The R-squared value is about `0.402`, so `age` explains about `40%` of the variation in car price, leaving about `60%` unexplained.

The average prediction is close to the average actual price. The actual mean is about `$19,318`, and the predicted mean is about `$19,339`, so the model is centered correctly on average.

However, the predictions are much less spread out than the real prices. Actual prices have a standard deviation of about `$13,040`, while predictions have a standard deviation of about `$8,243`. This means the model compresses predictions toward the middle: it tends to underestimate expensive cars and overestimate cheap cars.

The median prediction is also too high. The actual median price is about `$16,591`, but the predicted median is about `$21,466`, so for a typical car the model tends to predict too high.

The negative minimum prediction is a warning sign. The model predicts as low as about `-$7,673`, which is impossible for a car price. This happens because plain linear regression does not know that prices cannot be negative.

The maximum prediction is too low. The most expensive actual car is `$65,500`, but the model never predicts above about `$33,735`, so it does not capture high-priced cars well using only `age`.

Bottom line: `age` is useful, but not enough by itself. This model captures the general idea that newer cars tend to cost more, but it misses many other important factors like manufacturer, model, mileage, condition, fuel type, title status, and vehicle type.

### Polynomial regression using age

This cell fits a degree-2 polynomial regression model using only `age`, then reports the same performance and prediction summary statistics.

In [27]:
features = ["age"]

y_train = X_train["price"]
y_test = X_test["price"]

select_features = ColumnTransformer(
    transformers=[
        ("selected", "passthrough", features)
    ],
    remainder="drop"
)

poly_pipe = Pipeline([
    ("select_features", select_features),
    ("polynomial_features", PolynomialFeatures(degree=2, include_bias=False)),
    ("model", LinearRegression())
])

poly_pipe.fit(X_train, y_train)
poly_preds = poly_pipe.predict(X_test)

poly_performance = pd.Series({
    "mae": mean_absolute_error(y_test, poly_preds),
    "rmse": mean_squared_error(y_test, poly_preds) ** 0.5,
    "r2": r2_score(y_test, poly_preds),
})

poly_prediction_summary = pd.DataFrame({
    "y_test": y_test,
    "preds": poly_preds,
}).agg(["mean", "std", "median", "min", "max"])

print(poly_performance)
print(poly_prediction_summary)

mae     7241.872608
rmse    9550.166750
r2         0.463607
dtype: float64
              y_test         preds
mean    19318.045232  19348.064970
std     13039.824946   8862.626178
median  16591.000000  19550.424557
min      1000.000000   6539.210295
max     65500.000000  42915.266279


### Interpreting the polynomial model performance

These results are better than the simple `age` linear model.

Compared with the previous model, MAE improved from about `$7,915` to about `$7,242`, RMSE improved from about `$10,084` to about `$9,550`, and R-squared improved from about `0.402` to about `0.464`. This means the degree-2 polynomial model reduces the average error, reduces larger errors, and explains about `46%` of price variation instead of about `40%`.

The prediction summary also looks better. The minimum prediction is now positive, about `$6,539` instead of `-$7,673`. The maximum prediction increased to about `$42,915`, which is closer to the actual maximum of `$65,500`. The median prediction dropped from about `$21,466` to about `$19,550`, which is closer to the actual median of `$16,591`. The prediction standard deviation also increased from about `$8,243` to about `$8,863`, so the model is capturing a bit more spread in prices.

Bottom line: the polynomial model is modestly better. It still compresses predictions toward the middle and misses many price drivers, but adding the squared age term improved the fit.

### Simple linear regression using odometer

This cell fits a simple linear regression model using only `odometer`, then reports the same performance and prediction summary statistics.

In [28]:
features = ["odometer"]

y_train = X_train["price"]
y_test = X_test["price"]

select_features = ColumnTransformer(
    transformers=[
        ("selected", "passthrough", features)
    ],
    remainder="drop"
)

odometer_pipe = Pipeline([
    ("select_features", select_features),
    ("model", LinearRegression())
])

odometer_pipe.fit(X_train, y_train)
odometer_preds = odometer_pipe.predict(X_test)

odometer_performance = pd.Series({
    "mae": mean_absolute_error(y_test, odometer_preds),
    "rmse": mean_squared_error(y_test, odometer_preds) ** 0.5,
    "r2": r2_score(y_test, odometer_preds),
})

odometer_prediction_summary = pd.DataFrame({
    "y_test": y_test,
    "preds": odometer_preds,
}).agg(["mean", "std", "median", "min", "max"])

print(odometer_performance)
print(odometer_prediction_summary)

mae      8022.115216
rmse    10402.159069
r2          0.363633
dtype: float64
              y_test         preds
mean    19318.045232  19328.460169
std     13039.824946   7779.182323
median  16591.000000  19735.511459
min      1000.000000  -1660.280912
max     65500.000000  31420.551645


### Polynomial regression using odometer

This cell fits a degree-2 polynomial regression model using only `odometer`, then reports the same performance and prediction summary statistics.

In [29]:
features = ["odometer"]

y_train = X_train["price"]
y_test = X_test["price"]

select_features = ColumnTransformer(
    transformers=[
        ("selected", "passthrough", features)
    ],
    remainder="drop"
)

odometer_poly_pipe = Pipeline([
    ("select_features", select_features),
    ("polynomial_features", PolynomialFeatures(degree=2, include_bias=False)),
    ("model", LinearRegression())
])

odometer_poly_pipe.fit(X_train, y_train)
odometer_poly_preds = odometer_poly_pipe.predict(X_test)

odometer_poly_performance = pd.Series({
    "mae": mean_absolute_error(y_test, odometer_poly_preds),
    "rmse": mean_squared_error(y_test, odometer_poly_preds) ** 0.5,
    "r2": r2_score(y_test, odometer_poly_preds),
})

odometer_poly_prediction_summary = pd.DataFrame({
    "y_test": y_test,
    "preds": odometer_poly_preds,
}).agg(["mean", "std", "median", "min", "max"])

print(odometer_poly_performance)
print(odometer_poly_prediction_summary)

mae      7721.249220
rmse    10191.794283
r2          0.389111
dtype: float64
              y_test         preds
mean    19318.045232  19330.016394
std     13039.824946   8066.058018
median  16591.000000  17823.517454
min      1000.000000   8065.369976
max     65500.000000  35376.662376


### Polynomial regression using age and odometer

This cell fits a degree-2 polynomial regression model using both `age` and `odometer`, then reports the same performance and prediction summary statistics.

In [30]:
features = ["age", "odometer"]

y_train = X_train["price"]
y_test = X_test["price"]

select_features = ColumnTransformer(
    transformers=[
        ("selected", "passthrough", features)
    ],
    remainder="drop"
)

age_odometer_poly_pipe = Pipeline([
    ("select_features", select_features),
    ("polynomial_features", PolynomialFeatures(degree=2, include_bias=False)),
    ("model", LinearRegression())
])

age_odometer_poly_pipe.fit(X_train, y_train)
age_odometer_poly_preds = age_odometer_poly_pipe.predict(X_test)

age_odometer_poly_performance = pd.Series({
    "mae": mean_absolute_error(y_test, age_odometer_poly_preds),
    "rmse": mean_squared_error(y_test, age_odometer_poly_preds) ** 0.5,
    "r2": r2_score(y_test, age_odometer_poly_preds),
})

age_odometer_poly_prediction_summary = pd.DataFrame({
    "y_test": y_test,
    "preds": age_odometer_poly_preds,
}).agg(["mean", "std", "median", "min", "max"])

print(age_odometer_poly_performance)
print(age_odometer_poly_prediction_summary)

mae     6966.452088
rmse    9275.109345
r2         0.494060
dtype: float64
              y_test         preds
mean    19318.045232  19347.322907
std     13039.824946   9143.984814
median  16591.000000  18903.958755
min      1000.000000   3134.814869
max     65500.000000  40177.117703
